# MobileNet Batch Inference with TaskVine

This tutorial classifies 24 Wikimedia Commons images with a CPU-only MobileNetV2 model. Floability has already staged the images, model, labels, software environment, and TaskVine workers.

You will choose one TaskVine execution method and then follow the same workflow:

1. verify and batch the input images;
2. connect a TaskVine manager to the workers started by Floability;
3. declare the model, labels, and image batches;
4. submit distributed inference tasks;
5. collect and inspect predictions; and
6. save a JSON summary and contact sheet.

Run the notebook from top to bottom. To compare the other method, restart the kernel, change the option below, and run all cells again.


## Step 1 — Choose an execution method

Select one of these values:

- **python-task** — each ordinary TaskVine `PythonTask` creates its own ONNX Runtime session. This is simple and works well for longer functions whose setup cost is small.
- **stateful-serverless** — TaskVine's **Stateful Serverless Computing** model. A persistent `LibraryTask` loads MobileNet once, and `FunctionCall` tasks reuse that loaded session.

The default starts with the ordinary Python task path.


In [ ]:
# EXECUTION_MODE = "python-task"
EXECUTION_MODE = "stateful-serverless"

VALID_EXECUTION_MODES = {"python-task", "stateful-serverless"}
if EXECUTION_MODE not in VALID_EXECUTION_MODES:
    raise ValueError(
        f"EXECUTION_MODE must be one of {sorted(VALID_EXECUTION_MODES)}"
    )

BATCH_SIZE = 4
TOP_K = 5
LIBRARY_NAME = "mobilenetv2-inference"

print(f"Selected execution method: {EXECUTION_MODE}")


## Step 2 — Import packages and locate staged inputs

Floability runs this notebook inside the backpack's workflow directory. The paths below therefore refer to files staged from `data/data.yml`. Generated results go into `outputs/`.


In [ ]:
import hashlib
import json
import os
import shutil
import tempfile
import time
from collections import defaultdict
from pathlib import Path

import ndcctools.taskvine as vine
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

MODEL_PATH = Path("data/mobilenetv2-10.onnx")
LABELS_PATH = Path("data/imagenet-synset.txt")
IMAGE_DIR = Path("data/images")
MANIFEST_PATH = Path("data/image-manifest.json")
OUTPUT_DIR = Path("outputs")
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

for required_path in (MODEL_PATH, LABELS_PATH, IMAGE_DIR, MANIFEST_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Required staged input not found: {required_path}")

print(f"Model: {MODEL_PATH}")
print(f"Images: {IMAGE_DIR}")


## Step 3 — Define the worker-side inference functions

Both methods use identical resizing, normalization, inference, and top-five postprocessing.

The ordinary function loads the model inside every task. The serverless setup function loads it once per persistent library instance; subsequent function calls retrieve that shared session from the library.


In [ ]:
def classify_batch_cold(model_path, labels_path, batch_dir, top_k):
    """Load MobileNet in this PythonTask, then classify one microbatch."""
    import os
    import socket
    import time
    import uuid
    from pathlib import Path

    import numpy as np
    import onnxruntime as ort
    from PIL import Image

    started_at = time.perf_counter()
    session_options = ort.SessionOptions()
    session_options.intra_op_num_threads = 1
    session_options.inter_op_num_threads = 1
    session = ort.InferenceSession(
        model_path,
        sess_options=session_options,
        providers=["CPUExecutionProvider"],
    )
    with open(labels_path, encoding="utf-8") as labels_file:
        labels = [
            line.strip().split(" ", 1)[1]
            for line in labels_file
            if line.strip()
        ]
    loaded_at = time.perf_counter()

    predictions = []
    for image_path in sorted(Path(batch_dir).iterdir()):
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        with Image.open(image_path) as source_image:
            image = source_image.convert("RGB")
            width, height = image.size
            scale = 256.0 / min(width, height)
            resized = image.resize(
                (round(width * scale), round(height * scale)),
                Image.Resampling.BILINEAR,
            )
            left = (resized.width - 224) // 2
            top = (resized.height - 224) // 2
            cropped = resized.crop((left, top, left + 224, top + 224))

        image_array = np.asarray(cropped, dtype=np.float32) / 255.0
        image_array = (
            image_array - np.array([0.485, 0.456, 0.406], dtype=np.float32)
        ) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
        input_tensor = np.transpose(image_array, (2, 0, 1))[None, ...]

        scores = session.run(
            None,
            {session.get_inputs()[0].name: input_tensor},
        )[0].reshape(-1)
        probabilities = np.exp(scores - np.max(scores))
        probabilities /= probabilities.sum()
        best_indices = np.argsort(probabilities)[-top_k:][::-1]

        predictions.append(
            {
                "image": image_path.name,
                "top_predictions": [
                    {
                        "class_index": int(index),
                        "label": labels[index],
                        "probability": float(probabilities[index]),
                    }
                    for index in best_indices
                ],
            }
        )

    return {
        "predictions": predictions,
        "session_load_id": uuid.uuid4().hex[:8],
        "session_load_seconds": loaded_at - started_at,
        "task_seconds": time.perf_counter() - started_at,
        "hostname": socket.gethostname(),
        "pid": os.getpid(),
    }


In [ ]:
def load_mobilenet_library(model_path, labels_path):
    """Create shared state when a persistent library instance starts."""
    import os
    import socket
    import time
    import uuid

    import onnxruntime as ort

    started_at = time.perf_counter()
    session_options = ort.SessionOptions()
    session_options.intra_op_num_threads = 1
    session_options.inter_op_num_threads = 1
    session = ort.InferenceSession(
        model_path,
        sess_options=session_options,
        providers=["CPUExecutionProvider"],
    )
    with open(labels_path, encoding="utf-8") as labels_file:
        labels = [
            line.strip().split(" ", 1)[1]
            for line in labels_file
            if line.strip()
        ]

    return {
        "inference_session": session,
        "imagenet_labels": labels,
        "library_load_id": uuid.uuid4().hex[:8],
        "library_load_seconds": time.perf_counter() - started_at,
        "library_hostname": socket.gethostname(),
        "library_pid": os.getpid(),
    }


def classify_batch_stateful(batch_dir, top_k):
    """Classify a microbatch with the library's already-loaded session."""
    import os
    import time
    from pathlib import Path

    import numpy as np
    from ndcctools.taskvine.utils import load_variable_from_library
    from PIL import Image

    started_at = time.perf_counter()
    session = load_variable_from_library("inference_session")
    labels = load_variable_from_library("imagenet_labels")

    predictions = []
    for image_path in sorted(Path(batch_dir).iterdir()):
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue

        with Image.open(image_path) as source_image:
            image = source_image.convert("RGB")
            width, height = image.size
            scale = 256.0 / min(width, height)
            resized = image.resize(
                (round(width * scale), round(height * scale)),
                Image.Resampling.BILINEAR,
            )
            left = (resized.width - 224) // 2
            top = (resized.height - 224) // 2
            cropped = resized.crop((left, top, left + 224, top + 224))

        image_array = np.asarray(cropped, dtype=np.float32) / 255.0
        image_array = (
            image_array - np.array([0.485, 0.456, 0.406], dtype=np.float32)
        ) / np.array([0.229, 0.224, 0.225], dtype=np.float32)
        input_tensor = np.transpose(image_array, (2, 0, 1))[None, ...]

        scores = session.run(
            None,
            {session.get_inputs()[0].name: input_tensor},
        )[0].reshape(-1)
        probabilities = np.exp(scores - np.max(scores))
        probabilities /= probabilities.sum()
        best_indices = np.argsort(probabilities)[-top_k:][::-1]

        predictions.append(
            {
                "image": image_path.name,
                "top_predictions": [
                    {
                        "class_index": int(index),
                        "label": labels[index],
                        "probability": float(probabilities[index]),
                    }
                    for index in best_indices
                ],
            }
        )

    return {
        "predictions": predictions,
        "library_load_id": load_variable_from_library("library_load_id"),
        "library_load_seconds": load_variable_from_library(
            "library_load_seconds"
        ),
        "library_hostname": load_variable_from_library("library_hostname"),
        "library_pid": load_variable_from_library("library_pid"),
        "function_pid": os.getpid(),
        "call_seconds": time.perf_counter() - started_at,
    }


## Step 4 — Connect the TaskVine manager

Floability supplies a unique manager name and an allowed port range through environment variables. The workers launched by `vine_factory` discover this manager by name.


In [ ]:
def manager_ports():
    port_spec = os.environ.get("VINE_MANAGER_PORTS", "9123,9150")
    ports = [int(value.strip()) for value in port_spec.split(",") if value.strip()]
    if not ports:
        raise ValueError("VINE_MANAGER_PORTS does not contain a port")
    if len(ports) == 1:
        return ports[0]
    return [min(ports), max(ports)]


manager_name = os.environ.get("VINE_MANAGER_NAME")
if not manager_name:
    raise RuntimeError(
        "VINE_MANAGER_NAME is not set; start this notebook through Floability"
    )

manager = vine.Manager(port=manager_ports(), name=manager_name)
manager.tune("watch-library-logfiles", 1)

print(f"Manager name: {manager_name}")
print(f"Manager port: {manager.port}")


## Step 5 — Verify and divide the images

The manifest protects the tutorial dataset from accidental changes. The 24 verified images are copied into six deterministic four-image directories. Each directory becomes one TaskVine microbatch input.


In [ ]:
image_paths = sorted(
    path for path in IMAGE_DIR.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS
)
if len(image_paths) != 24:
    raise RuntimeError(f"Expected 24 staged images; found {len(image_paths)}")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
manifest_by_file = {item["file"]: item for item in manifest["images"]}
if set(manifest_by_file) != {path.name for path in image_paths}:
    raise RuntimeError("The staged image set does not match image-manifest.json")

for image_path in image_paths:
    with image_path.open("rb") as image_file:
        actual_sha256 = hashlib.file_digest(image_file, "sha256").hexdigest()
    expected_sha256 = manifest_by_file[image_path.name]["local_sha256"]
    if actual_sha256 != expected_sha256:
        raise RuntimeError(f"SHA-256 mismatch for {image_path.name}")

temporary_batch_root = Path(
    tempfile.mkdtemp(prefix="mobilenet-notebook-batches-", dir=".")
)
batch_paths = []
for batch_number, start in enumerate(range(0, len(image_paths), BATCH_SIZE)):
    batch_path = temporary_batch_root / f"batch-{batch_number:02d}"
    batch_path.mkdir()
    for image_path in image_paths[start : start + BATCH_SIZE]:
        shutil.copy2(image_path, batch_path / image_path.name)
    batch_paths.append(batch_path)

print(f"Verified images: {len(image_paths)}")
print(f"Microbatches: {len(batch_paths)} × {BATCH_SIZE} images")


## Step 6 — Declare files for TaskVine

Declaring a file tells TaskVine what must be available in a worker sandbox. The model and labels are cached and reused. Each task receives one declared image directory as `batch/`.


In [ ]:
declared_model = manager.declare_file(str(MODEL_PATH), cache=True)
declared_labels = manager.declare_file(str(LABELS_PATH), cache=True)
declared_batches = {
    batch_path: manager.declare_file(str(batch_path), cache=True)
    for batch_path in batch_paths
}

print("Declared the model, labels, and image microbatches")


## Step 7 — Configure the selected execution method

For ordinary Python tasks, no persistent service is needed.

For stateful serverless execution, TaskVine creates a library from the inference function. The library context loads the model and labels once per library instance. `add_env=False` reuses the software environment already distributed by Floability, while direct execution keeps each call in the persistent library process.


In [ ]:
if EXECUTION_MODE == "stateful-serverless":
    library = manager.create_library_from_functions(
        LIBRARY_NAME,
        classify_batch_stateful,
        add_env=False,
        exec_mode="direct",
        library_context_info=[
            load_mobilenet_library,
            ["model.onnx", "labels.txt"],
            {},
        ],
    )
    library.add_input(declared_model, "model.onnx")
    library.add_input(declared_labels, "labels.txt")
    library.set_cores(1)
    library.set_function_slots(1)
    manager.install_library(library)
    print(f"Installed persistent library: {LIBRARY_NAME}")
else:
    print("PythonTask mode needs no persistent library")


## Step 8 — Submit one task per microbatch

The branch below changes only the TaskVine task type. Both methods receive the same image batches, model, labels, core request, and inference settings.


In [ ]:
task_batches = {}
started_at = time.perf_counter()

for batch_path in batch_paths:
    if EXECUTION_MODE == "python-task":
        task = vine.PythonTask(
            classify_batch_cold,
            "model.onnx",
            "labels.txt",
            "batch",
            TOP_K,
        )
        task.add_input(declared_model, "model.onnx")
        task.add_input(declared_labels, "labels.txt")
    else:
        task = vine.FunctionCall(
            LIBRARY_NAME,
            "classify_batch_stateful",
            "batch",
            TOP_K,
        )

    task.add_input(declared_batches[batch_path], "batch")
    task.set_cores(1)
    task_id = manager.submit(task)
    task_batches[task_id] = batch_path.name

print(f"Submitted {len(task_batches)} tasks using {EXECUTION_MODE}")


## Step 9 — Collect distributed results

The manager waits for completed tasks, records their worker addresses, and stops immediately if any task fails. In serverless mode, it also checks that each function ran inside its persistent library process.


In [ ]:
results = []
failures = []

while not manager.empty():
    completed = manager.wait(5)
    if not completed:
        continue

    if not completed.successful():
        failures.append((completed.id, completed.result))
        print(f"FAILED task={completed.id} result={completed.result}")
        continue

    result = completed.output
    if EXECUTION_MODE == "stateful-serverless":
        if result["function_pid"] != result["library_pid"]:
            raise RuntimeError(
                "FunctionCall did not execute inside the persistent library process"
            )
        execution_detail = f"load_id={result['library_load_id']}"
    else:
        execution_detail = f"session={result['session_load_id']}"

    result["task_id"] = completed.id
    result["batch"] = task_batches[completed.id]
    result["worker_address"] = completed.addrport
    results.append(result)

    print(
        f"task={completed.id} batch={result['batch']} "
        f"{execution_detail} worker={completed.addrport}"
    )

if failures:
    raise RuntimeError(f"Inference task failures: {failures}")
if len(results) != len(batch_paths):
    raise RuntimeError(
        f"Expected {len(batch_paths)} task results; received {len(results)}"
    )

elapsed_seconds = time.perf_counter() - started_at
print(f"All tasks completed in {elapsed_seconds:.2f} seconds")


## Step 10 — Validate and inspect predictions

Every image must appear exactly once. The table shows the most likely ImageNet class for each image. In stateful mode, repeated library-load IDs demonstrate that several calls reused one initialized model.


In [ ]:
predictions_by_image = {
    prediction["image"]: prediction
    for task_result in results
    for prediction in task_result["predictions"]
}
if set(predictions_by_image) != {path.name for path in image_paths}:
    raise RuntimeError("Results do not contain exactly the 24 input images")

if EXECUTION_MODE == "stateful-serverless":
    library_reuse = defaultdict(list)
    for result in results:
        library_reuse[result["library_load_id"]].append(result["batch"])
    if len(library_reuse) >= len(results):
        raise RuntimeError("Every FunctionCall created a new library instance")
    print(f"Persistent library instances: {len(library_reuse)}")
    for load_id, batches in sorted(library_reuse.items()):
        print(f"  {load_id}: {len(batches)} microbatch(es)")
else:
    library_reuse = {}
    print(f"Independent ONNX sessions: {len(results)}")

print()
print(f"{'Image':<27} {'Top prediction':<40} Confidence")
print("-" * 82)
for image_name in sorted(predictions_by_image):
    top_prediction = predictions_by_image[image_name]["top_predictions"][0]
    print(
        f"{image_name:<27} "
        f"{top_prediction['label'][:39]:<40} "
        f"{top_prediction['probability']:.1%}"
    )


## Step 11 — Save and visualize the run

The final cell writes all predictions and worker metadata to JSON. It also creates a contact sheet labeled with each image's top prediction and confidence.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary = {
    "execution_mode": EXECUTION_MODE,
    "image_count": len(image_paths),
    "batch_size": BATCH_SIZE,
    "batch_count": len(batch_paths),
    "elapsed_seconds": elapsed_seconds,
    "distinct_library_loads": (
        len(library_reuse) if EXECUTION_MODE == "stateful-serverless" else None
    ),
    "task_results": results,
}
summary_path = OUTPUT_DIR / f"{EXECUTION_MODE}-summary.json"
summary_path.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

tile_width, tile_height = 240, 220
caption_height = 52
contact_sheet = Image.new("RGB", (tile_width * 4, tile_height * 6), "white")
draw = ImageDraw.Draw(contact_sheet)
font = ImageFont.load_default()

for index, image_path in enumerate(image_paths):
    with Image.open(image_path) as source_image:
        thumbnail = source_image.convert("RGB")
        thumbnail.thumbnail((tile_width - 12, tile_height - caption_height - 12))

    x = (index % 4) * tile_width
    y = (index // 4) * tile_height
    image_x = x + (tile_width - thumbnail.width) // 2
    image_y = y + 6
    contact_sheet.paste(thumbnail, (image_x, image_y))

    prediction = predictions_by_image[image_path.name]["top_predictions"][0]
    caption = (
        f"{image_path.stem[:24]}\n"
        f"{prediction['label'][:34]}\n"
        f"{prediction['probability']:.1%}"
    )
    draw.multiline_text(
        (x + 6, y + tile_height - caption_height),
        caption,
        fill="black",
        font=font,
        spacing=2,
    )

contact_sheet_path = OUTPUT_DIR / f"{EXECUTION_MODE}-contact-sheet.jpg"
contact_sheet.save(contact_sheet_path, quality=90)

print("=" * 72)
print("MOBILENET BATCH INFERENCE COMPLETE")
print(f"Execution mode: {EXECUTION_MODE}")
print(f"Validated images: {len(predictions_by_image)}")
print(f"Elapsed time: {elapsed_seconds:.2f} seconds")
print(f"Results: {summary_path}")
print(f"Contact sheet: {contact_sheet_path}")
print("=" * 72)

display(contact_sheet)


## Interpreting the two methods

The prediction results should be the same in both modes; the difference is model lifecycle.

- **PythonTask:** simple task isolation, but each microbatch pays the model initialization cost.
- **Stateful Serverless Computing:** each library instance pays initialization once and reuses its ONNX session across later function calls.

This 24-image run is a functional demonstration, not a formal performance benchmark. Worker startup, scheduling, transfer caching, and batch-system queue time can affect wall-clock measurements.
